### Preparation

In [ ]:
pip install transformers datasets evaluate torch

In [ ]:
import os
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_DISABLED"] = "true"

### Training Logic V1

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback
from datasets import load_dataset
import evaluate

# Load dataset SQuAD v2
dataset = load_dataset("rajpurkar/squad_v2")

# Load model and tokenizer
model_name = "bert-base-uncased"
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Preprocessing data
def preprocess_function(examples):
    questions = examples["question"]
    contexts = examples["context"]
    answers = examples["answers"]

    inputs = tokenizer(questions, contexts, max_length=256, truncation=True, 
                       padding="max_length", return_offsets_mapping=True)

    start_positions, end_positions = [], []
    for i, offsets in enumerate(inputs["offset_mapping"]):
        if len(answers[i]["text"]) == 0:  # Handle no answer case in SQuAD v2
            start_positions.append(0)
            end_positions.append(0)
        else:
            answer = answers[i]["text"][0]
            start_char = answers[i]["answer_start"][0]
            end_char = start_char + len(answer)

            start_token, end_token = 0, 0
            for j, offset in enumerate(offsets):
                if offset[0] <= start_char and offset[1] > start_char:
                    start_token = j
                if offset[0] < end_char and offset[1] >= end_char:
                    end_token = j

            start_positions.append(start_token)
            end_positions.append(end_token)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    inputs.pop("offset_mapping")  # Not needed for training
    return inputs

# Tokenizing dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

# Load evaluation metrics using the evaluate library
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
exact_match_metric = evaluate.load("exact_match")

# Training configuration
training_args = TrainingArguments(
    output_dir="./distilbert-squad",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=100,
    save_steps=500,
    warmup_steps=500,
    fp16=True,  # Mixed precision training (if GPU supports it)
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
)

# Start training
trainer.train()

# Save trained model
trainer.save_model("./bert-squad")


### Training Logic V2 (For Huggingface)

In [ ]:
squad_v2 = True
model_checkpoint = "albert-base-v2"
batch_size = 16

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset
datasets = load_dataset("squad_v2" if squad_v2 else "squad")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

max_length = 384
doc_stride = 128
pad_on_right = tokenizer.padding_side == "right"

print("Dataset:", datasets)
print("Tokenizer:", type(tokenizer).__name__)

In [ ]:
def preprocess_function(examples):
    examples["question"] = [q.lstrip() for q in examples["question"]]
    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])
            token_start_index = 0
            while sequence_ids[token_start_index] != (1 if pad_on_right else 0):
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != (1 if pad_on_right else 0):
                token_end_index -= 1
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)
    return tokenized_examples

# Tokenized all dataset
tokenized_datasets = datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=datasets["train"].column_names
)
print("Preprocessing selesai.")

In [ ]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator

model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

args = TrainingArguments(
    f"{model_checkpoint.split('/')[-1]}-finetuned-squad-v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=1,
    weight_decay=0.01,
)

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=default_data_collator,
    tokenizer=tokenizer,
)

trainer.train()

# Save model
trainer.save_model("albert-squadv2-trained")

### Evaluasi

In [ ]:
squad_v2 = False
model_dirs = [
    "../bert-squad",
    "../distilbert-squad",
    "../albert-squad"
]

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer
import numpy as np
from collections import defaultdict
import evaluate

# Load dataset
squad_v1 = load_dataset("squad_v2" if squad_v2 else "squad")

# Preprocess function (shared for all models)
max_length = 384
doc_stride = 128

def preprocess_function(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized

# Postprocess function
def postprocess_qa_predictions(examples, features, predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        valid_answers = []
        context = example["context"]

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                        or end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    text = context[start_char:end_char]
                    score = start_logits[start_index] + end_logits[end_index]
                    valid_answers.append({"score": score, "text": text})

        best_answer = max(valid_answers, key=lambda x: x["score"]) if valid_answers else {"text": ""}
        predictions[example["id"]] = best_answer["text"]

    return predictions

# Evaluate multiple models
metric = evaluate.load("squad_v2" if squad_v2 else "squad")

results_all = {}

for model_dir in model_dirs:
    print(f"\n🔹 Evaluating {model_dir}...")

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForQuestionAnswering.from_pretrained(model_dir)

    # Tokenize validation set
    tokenized_val = squad_v1["validation"].map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=squad_v1["validation"].column_names
    )

    # Run predictions
    trainer = Trainer(model=model)
    raw_predictions = trainer.predict(tokenized_val)
    start_logits, end_logits = raw_predictions.predictions

    # Postprocess
    final_predictions = postprocess_qa_predictions(
        squad_v1["validation"], tokenized_val, (start_logits, end_logits)
    )

    # Evaluate
    results = metric.compute(
        predictions=[{"id": k, "prediction_text": v} for k, v in final_predictions.items()],
        references=[{"id": ex["id"], "answers": ex["answers"]} for ex in squad_v1["validation"]]
    )

    results_all[model_dir] = results
    print(f"Results for {model_dir}: {results}")

# Print comparison
print("\n📊 Final Comparison:")
for model_dir, res in results_all.items():
    print(f"{model_dir}: Exact Match = {res['exact_match']:.2f}, F1 = {res['f1']:.2f}")
